# Bigram language Model

In [1]:
import torch
import torch.nn as nn
import math
import random
from typing import List, Tuple, Dict

# Data Preprocessing

In [13]:
text= """Once in a small village surrounded by green hills,
 there lived a young girl named Mira who loved fixing broken things.
  While others threw away old radios, clocks, and toys, Mira collected
   them and spent hours trying to repair them. One summer, a heavy storm
    damaged the village water pump, leaving everyone without clean water.
     The villagers worried because the nearest mechanic lived far away.
      Mira quietly examined the machine, cleaned the rusted parts, and
      used pieces from old tools she had saved. After several hours of work,
       the pump started running again, and water flowed through the village
        once more. Everyone was amazed that such a young girl could solve
         such a big problem. From that day, the villagers learned that curiosity,
          patience, and knowledge can make even ordinary people extraordinary."""

chars = sorted(list(set(text)))
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

print(stoi)
vocab_size = len(chars)
print(vocab_size)


{'\n': 0, ' ': 1, ',': 2, '.': 3, 'A': 4, 'E': 5, 'F': 6, 'M': 7, 'O': 8, 'T': 9, 'W': 10, 'a': 11, 'b': 12, 'c': 13, 'd': 14, 'e': 15, 'f': 16, 'g': 17, 'h': 18, 'i': 19, 'k': 20, 'l': 21, 'm': 22, 'n': 23, 'o': 24, 'p': 25, 'q': 26, 'r': 27, 's': 28, 't': 29, 'u': 30, 'v': 31, 'w': 32, 'x': 33, 'y': 34, 'z': 35}
36


In [14]:
# Creating training data
# x = Currect character
# y = next character

xs = []
ys = []

for ch1, ch2 in zip(text, text[1:]):
  xs.append(stoi[ch1])
  ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
print(xs.shape)

torch.Size([867])


In [15]:
import torch.nn.functional as F

# Building Model

In [21]:
class BigramNeuralNet(nn.Module):
  def __init__(self, vocab_size,embed_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.linear = nn.Linear(embed_dim, vocab_size)

  def forward(self, context):
    embedded = self.embedding(context)
    logits = self.linear(embedded)
    return logits

model = BigramNeuralNet(vocab_size, embed_dim=10)

In [22]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [29]:
for epoch in range(10000):
  logits = model(xs)
  loss = F.cross_entropy(logits, ys)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 50 == 0:
    print(f"Epoch: {epoch}  Loss: {loss.item()}"  )

Epoch: 0  Loss: 2.111438274383545
Epoch: 50  Loss: 2.1114397048950195
Epoch: 100  Loss: 2.111445188522339
Epoch: 150  Loss: 2.1114449501037598
Epoch: 200  Loss: 2.1114449501037598
Epoch: 250  Loss: 2.1114511489868164
Epoch: 300  Loss: 2.1114578247070312
Epoch: 350  Loss: 2.111438035964966
Epoch: 400  Loss: 2.1114425659179688
Epoch: 450  Loss: 2.111445665359497
Epoch: 500  Loss: 2.111438751220703
Epoch: 550  Loss: 2.111466407775879
Epoch: 600  Loss: 2.111443042755127
Epoch: 650  Loss: 2.111438035964966
Epoch: 700  Loss: 2.1114461421966553
Epoch: 750  Loss: 2.1114377975463867
Epoch: 800  Loss: 2.1114413738250732
Epoch: 850  Loss: 2.1114673614501953
Epoch: 900  Loss: 2.1114375591278076
Epoch: 950  Loss: 2.1114463806152344
Epoch: 1000  Loss: 2.1114912033081055
Epoch: 1050  Loss: 2.111438751220703
Epoch: 1100  Loss: 2.1114373207092285
Epoch: 1150  Loss: 2.111442804336548
Epoch: 1200  Loss: 2.111441135406494
Epoch: 1250  Loss: 2.1114614009857178
Epoch: 1300  Loss: 2.111438512802124
Epoch: 13

# Generate Text

In [38]:
idx = torch.tensor([stoi['h']])
for _ in range(20):
  logits = model(idx)
  probs = F.softmax(logits, dim=-1)

  next_token_idx = torch.multinomial(probs.squeeze(), num_samples=1)
  print(itos[next_token_idx.item()], end="")

  idx = next_token_idx.unsqueeze(0)

evigeagenaved   prs 

In [39]:
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

memory_bytes = total_params * 4

print("Approx Memory:", memory_bytes / 1024, "KB")

Total Parameters: 756
Approx Memory: 2.953125 KB


In [40]:
for name, param in model.named_parameters():

    print(name)

    print("Shape:", param.shape)

    print("Parameters:", param.numel())

    print("Memory:", param.numel() * param.element_size(), "bytes")

embedding.weight
Shape: torch.Size([36, 10])
Parameters: 360
Memory: 1440 bytes
linear.weight
Shape: torch.Size([36, 10])
Parameters: 360
Memory: 1440 bytes
linear.bias
Shape: torch.Size([36])
Parameters: 36
Memory: 144 bytes


# Trigram Model

In [41]:
# Dataset

xs = []
ys = []

for ch1, ch2, ch3 in zip(text, text[1:], text[2:]):

    xs.append([stoi[ch1], stoi[ch2]])

    ys.append(stoi[ch3])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(xs.shape)
print(ys.shape)

torch.Size([866, 2])
torch.Size([866])


# Model

In [42]:
class TrigramNN(nn.Module):

    def __init__(self, vocab_size, embed_dim):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.linear = nn.Linear(embed_dim * 2, vocab_size)

    def forward(self, x):

        # x shape = (batch, 2)

        emb = self.embedding(x)

        # emb shape = (batch, 2, embed_dim)

        emb = emb.view(emb.shape[0], -1)

        # shape = (batch, 2*embed_dim)

        logits = self.linear(emb)

        return logits

In [43]:
model = TrigramNN(vocab_size, embed_dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1000):

    logits = model(xs)

    loss = F.cross_entropy(logits, ys)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(epoch, loss.item())

0 3.7634692192077637
100 1.7066811323165894
200 1.567625880241394
300 1.5376931428909302
400 1.5250874757766724
500 1.5193860530853271
600 1.5165139436721802
700 1.5147850513458252
800 1.513582706451416
900 1.5127052068710327


# Generate text

In [44]:
context = ['h', 'e']

generated = "he"

for _ in range(100):

    x = torch.tensor([[stoi[context[0]], stoi[context[1]]]])

    logits = model(x)

    probs = F.softmax(logits, dim=-1)

    idx = torch.multinomial(probs, 1).item()

    next_char = itos[idx]

    generated += next_char

    context = [context[1], next_char]

print(generated)

he village macleaned ordiry onerm
   n way.
 tharticlla ra the nary.
 t din  pieclecllagep exine wat s


In [45]:
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)
memory_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

print("Memory:", memory_bytes, "bytes")

print("Memory KB:", memory_bytes / 1024)

print("Memory MB:", memory_bytes / (1024**2))
for name, param in model.named_parameters():

    print(name)

    print("Shape:", tuple(param.shape))

    print("Parameters:", param.numel())

    print("Memory:", param.numel() * param.element_size(), "bytes")

    print()

Total Parameters: 1764
Memory: 7056 bytes
Memory KB: 6.890625
Memory MB: 0.0067291259765625
embedding.weight
Shape: (36, 16)
Parameters: 576
Memory: 2304 bytes

linear.weight
Shape: (36, 32)
Parameters: 1152
Memory: 4608 bytes

linear.bias
Shape: (36,)
Parameters: 36
Memory: 144 bytes

